<a href="https://colab.research.google.com/github/himanshusar123/-Machine-Learning-Quiz-Classification-or-Regression-/blob/main/Day_3_Harbinger_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏢 Day 3 – RetailMax HR Automation System Version 3.0
## Theme: "Migrating from CSV to SQL Database"

### Story Continuation
#### 9:00 AM – Morning Stand-up Meeting
The Project Manager enters.

> "Good morning team. Excellent work yesterday. HR is successfully using our application. We currently have 50 employee records. However, RetailMax is expanding rapidly. By next quarter, we'll have more than **50,000 employees** across India. During testing, the IT Architecture Team identified several problems with our CSV-based solution."

**Problems with CSV files:**
- Slow linear searches ($O(N)$ lookup)
- Risk of duplicate records (no primary keys)
- Race conditions (multiple users editing concurrently)
- No support for relationships (joins)
- Security and access control limitations
- Higher risk of file corruption

> "Management has approved migration to a relational database. Today, we migrate from CSV to SQLite database!"

## Sprint 1 – Why CSV Is No Longer Enough
As the organization scales from 50 to 50,000+ employees, reading a raw CSV row-by-row (linear search) becomes extremely slow and inefficient. Relational databases like **SQLite** provide fast queries, reliable storage, data consistency constraints, and structured schema support.

## Sprint 2 – Creating the Database
Let's create the company's first HR database using Python's built-in `sqlite3` module.

In [ ]:
import sqlite3

# Connect to database (creates retailmax.db if it doesn't exist)
connection = sqlite3.connect("retailmax.db")
print("Database Connected Successfully")

### Creating the Employee Table
Now we initialize a cursor to execute SQL commands and create the `employees` table with columns: `employee_id` (Primary Key), `name`, `department`, `designation`, and `salary`.

In [ ]:
cursor = connection.cursor()

# Create the table
cursor.execute("""
CREATE TABLE IF NOT EXISTS employees (
    employee_id INTEGER PRIMARY KEY,
    name TEXT,
    department TEXT,
    designation TEXT,
    salary REAL
)
""")

connection.commit()
print("Employee Table Created Successfully")

## Sprint 3 – Migrating CSV to Database
HR already has 50 employee records saved in `employees.csv`. We will read these records and migrate them into our new SQLite database. Since our source CSV has only 4 columns (ID, Name, Department, Salary) and the database has 5 columns (including Designation), we map 4 columns to 5 by defaulting `designation` to `"Associate"` during migration.

In [ ]:
import csv
import os

filename = "employees.csv"

if not os.path.exists(filename):
    print(f"Error: {filename} not found. Please verify the file path.")
else:
    with open(filename, "r", newline="", encoding="utf-8") as file:
        reader = csv.reader(file)
        rows = list(reader)
    
    print(f"Loaded {len(rows)} records from CSV.")
    
    inserted_count = 0
    for row in rows:
        if not row:
            continue
        
        # Handle CSVs with 4 columns vs 5 columns
        if len(row) == 4:
            emp_id, name, dept, salary = row
            designation = "Associate"  # Default value for database schema compatibility
        elif len(row) == 5:
            emp_id, name, dept, designation, salary = row
        else:
            print(f"Skipping malformed row: {row}")
            continue
            
        try:
            # Insert records OR ignore if primary key employee_id already exists
            cursor.execute("""
            INSERT OR IGNORE INTO employees (employee_id, name, department, designation, salary)
            VALUES (?, ?, ?, ?, ?)
            """, (int(emp_id), name, dept, designation, float(salary)))
            
            if cursor.rowcount > 0:
                inserted_count += 1
        except Exception as e:
            print(f"Error inserting employee ID {emp_id}: {e}")

    connection.commit()
    print(f"Successfully migrated {inserted_count} new employees into the database.")

## Sprint 4 – Display Employees
Let's fetch all records from the `employees` table and display them in a clean, formatted table layout.

In [ ]:
cursor.execute("SELECT * FROM employees")
employees = cursor.fetchall()

print("=" * 80)
print(f"{'ID':<10}{'Name':<25}{'Department':<20}{'Designation':<15}{'Salary':>10}")
print("=" * 80)
for emp in employees:
    print(f"{emp[0]:<10}{emp[1]:<25}{emp[2]:<20}{emp[3]:<15}{emp[4]:>10.2f}")
print("=" * 80)

## Sprint 5 – Search Employee
Search for a specific employee by ID using a **parameterized query** to prevent SQL injection.

In [ ]:
emp_id = input("Enter Employee ID to search: ")

# Parameterized query protects against SQL injection
cursor.execute("""
SELECT * 
FROM employees 
WHERE employee_id = ?
""", (emp_id,))

employee = cursor.fetchone()

if employee:
    print("\nEmployee Found")
    print("-" * 40)
    print(f"ID          : {employee[0]}")
    print(f"Name        : {employee[1]}")
    print(f"Department  : {employee[2]}")
    print(f"Designation : {employee[3]}")
    print(f"Salary      : {employee[4]:.2f}")
    print("-" * 40)
else:
    print("Employee Not Found")

## Sprint 6 – Add Employee
Insert a new employee record using parameterized input.

In [ ]:
print("Adding a new employee to RetailMax database:")
try:
    employee_id = int(input("Enter Employee ID: "))
    name = input("Enter Name: ")
    department = input("Enter Department: ")
    designation = input("Enter Designation: ")
    salary = float(input("Enter Salary: "))

    cursor.execute("""
    INSERT INTO employees (employee_id, name, department, designation, salary)
    VALUES (?, ?, ?, ?, ?)
    """, (employee_id, name, department, designation, salary))
    
    connection.commit()
    print("Employee Added Successfully!")
except sqlite3.IntegrityError:
    print("Error: An employee with this ID already exists (Primary Key Violation).")
except ValueError:
    print("Error: Invalid input. ID must be an integer, and Salary must be a number.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

## Sprint 7 – Update Employee
Update an employee's salary record using the unique `employee_id` primary key.

In [ ]:
try:
    employee_id = int(input("Enter Employee ID to update: "))
    new_salary = float(input("Enter New Salary: "))

    # Check if employee exists first
    cursor.execute("SELECT name FROM employees WHERE employee_id = ?", (employee_id,))
    if cursor.fetchone():
        cursor.execute("""
        UPDATE employees
        SET salary = ?
        WHERE employee_id = ?
        """, (new_salary, employee_id))
        connection.commit()
        print("Employee Salary Updated Successfully!")
    else:
        print("Employee ID not found.")
except ValueError:
    print("Error: Invalid input. ID and Salary must be numerical values.")

## Sprint 8 – Delete Employee
Remove an employee from the database by ID.

In [ ]:
try:
    employee_id = int(input("Enter Employee ID to delete: "))

    # Check if employee exists first
    cursor.execute("SELECT name FROM employees WHERE employee_id = ?", (employee_id,))
    employee = cursor.fetchone()
    if employee:
        cursor.execute("""
        DELETE FROM employees
        WHERE employee_id = ?
        """, (employee_id,))
        connection.commit()
        print(f"Employee {employee[0]} Deleted Successfully!")
    else:
        print("Employee ID not found.")
except ValueError:
    print("Error: Invalid ID. Please enter an integer.")

## Sprint 9 – HR Analytics
Relational databases are optimized to compute aggregate analytics on millions of records using built-in SQL functions.

In [ ]:
print("=" * 40)
print("           RetailMax HR Insights")
print("=" * 40)

# Total Employees
cursor.execute("SELECT COUNT(*) FROM employees")
total_emp = cursor.fetchone()[0]
print(f"Total Employees : {total_emp}")

# Average Salary
cursor.execute("SELECT AVG(salary) FROM employees")
avg_sal = cursor.fetchone()[0]
print(f"Average Salary  : {avg_sal:.2f}" if avg_sal else "Average Salary  : 0.00")

# Highest Salary
cursor.execute("SELECT MAX(salary) FROM employees")
max_sal = cursor.fetchone()[0]
print(f"Highest Salary  : {max_sal:.2f}" if max_sal else "Highest Salary  : 0.00")
print("-" * 40)

# Finance Employees List
print("Finance Department Employees:")
cursor.execute("SELECT * FROM employees WHERE department = 'Finance'")
finance_employees = cursor.fetchall()
for row in finance_employees:
    print(f" - ID: {row[0]}, Name: {row[1]}, Designation: {row[3]}, Salary: {row[4]:.2f}")

## Sprint 10 – Final Application
This combines all functions into an interactive menu loop. When running in a Jupyter Notebook, make sure to exit via option `8` to properly free inputs.

In [ ]:
def main_app():
    while True:
        print("\n" + "=" * 50)
        print("    RetailMax HR Automation System Version 3.0")
        print("=" * 50)
        print(" 1. View Employees")
        print(" 2. Search Employee")
        print(" 3. Add Employee")
        print(" 4. Update Employee")
        print(" 5. Delete Employee")
        print(" 6. Department Report")
        print(" 7. Salary Analytics")
        print(" 8. Exit")
        print("=" * 50)
        
        choice = input("Enter choice (1-8): ").strip()
        
        if choice == '1':
            cursor.execute("SELECT * FROM employees")
            records = cursor.fetchall()
            print("\n" + "-" * 75)
            print(f"{'ID':<10}{'Name':<25}{'Department':<20}{'Salary':>10}")
            print("-" * 75)
            for row in records:
                print(f"{row[0]:<10}{row[1]:<25}{row[2]:<20}{row[4]:>10.2f}")
            print("-" * 75)
            
        elif choice == '2':
            search_id = input("Enter Employee ID to search: ").strip()
            cursor.execute("SELECT * FROM employees WHERE employee_id = ?", (search_id,))
            row = cursor.fetchone()
            if row:
                print(f"\nID: {row[0]}\nName: {row[1]}\nDepartment: {row[2]}\nDesignation: {row[3]}\nSalary: {row[4]:.2f}")
            else:
                print("Employee not found.")
                
        elif choice == '3':
            try:
                emp_id = int(input("Enter Employee ID: "))
                name = input("Enter Name: ").strip()
                dept = input("Enter Department: ").strip()
                desig = input("Enter Designation: ").strip()
                salary = float(input("Enter Salary: "))
                
                cursor.execute("""
                INSERT INTO employees (employee_id, name, department, designation, salary)
                VALUES (?, ?, ?, ?, ?)
                """, (emp_id, name, dept, desig, salary))
                connection.commit()
                print("Employee added successfully.")
            except sqlite3.IntegrityError:
                print("Error: Employee ID already exists.")
            except ValueError:
                print("Error: Invalid values entered.")
                
        elif choice == '4':
            try:
                emp_id = int(input("Enter Employee ID to update: "))
                cursor.execute("SELECT name, salary FROM employees WHERE employee_id = ?", (emp_id,))
                row = cursor.fetchone()
                if row:
                    print(f"Current salary for {row[0]} is {row[1]:.2f}")
                    new_salary = float(input("Enter New Salary: "))
                    cursor.execute("UPDATE employees SET salary = ? WHERE employee_id = ?", (new_salary, emp_id))
                    connection.commit()
                    print("Salary updated successfully.")
                else:
                    print("Employee not found.")
            except ValueError:
                print("Error: Invalid values entered.")
                
        elif choice == '5':
            try:
                emp_id = int(input("Enter Employee ID to delete: "))
                cursor.execute("SELECT name FROM employees WHERE employee_id = ?", (emp_id,))
                row = cursor.fetchone()
                if row:
                    confirm = input(f"Delete employee {row[0]}? (y/n): ").strip().lower()
                    if confirm == 'y':
                        cursor.execute("DELETE FROM employees WHERE employee_id = ?", (emp_id,))
                        connection.commit()
                        print("Employee deleted successfully.")
                else:
                    print("Employee not found.")
            except ValueError:
                print("Error: Invalid ID.")
                
        elif choice == '6':
            dept_name = input("Enter Department Name: ").strip()
            cursor.execute("SELECT * FROM employees WHERE department LIKE ?", (dept_name,))
            records = cursor.fetchall()
            if records:
                print(f"\nEmployees in {dept_name}:")
                for r in records:
                    print(f" - ID: {r[0]}, Name: {r[1]}, Designation: {r[3]}, Salary: {r[4]:.2f}")
            else:
                print(f"No employees found in department '{dept_name}'.")
                
        elif choice == '7':
            cursor.execute("SELECT COUNT(*), AVG(salary), MAX(salary) FROM employees")
            stats = cursor.fetchone()
            print("\n" + "-" * 40)
            print("         Salary Insights")
            print("-" * 40)
            print(f"Total Employees : {stats[0]}")
            print(f"Average Salary  : {stats[1]:.2f}" if stats[1] else "Average Salary  : 0.00")
            print(f"Highest Salary  : {stats[2]:.2f}" if stats[2] else "Highest Salary  : 0.00")
            print("-" * 40)
            
        elif choice == '8':
            print("Exiting... Goodbye!")
            break
        else:
            print("Invalid choice. Choose 1-8.")

# Run the main application
main_app()